# Chinook Source Exploration

This notebook examines the original Chinook CSV files before creating the Raw, Clean, and Mart tables.

## Objectives

- Confirm the correct source folder
- Check the available CSV files
- Preview the important datasets
- Review row counts and column structures
- Identify the table relationships needed for the dimensional model

**Source path:**

`/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/`

In [0]:
LIST '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/';

## Source File Groups

The source files are grouped according to their role in the project:

- **Customer:** Customer.csv
- **Sales:** Invoice.csv and InvoiceLine.csv
- **Employee:** Employee.csv
- **Music:** Track.csv, Album.csv, Artist.csv, Genre.csv, and MediaType.csv
- **Playlist:** Playlist.csv and PlaylistTrack.csv

The files are normalized, which means related information is stored in separate tables and connected through IDs.

## Customer Data

The Customer file contains customer identity, contact information, location, and the assigned support representative.

In [0]:
%sql
SELECT *
FROM read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/Customer.csv',
    format => 'csv',
    header => true,
    inferSchema => true
)
LIMIT 10;

## Invoice Data

The Invoice file contains one row per invoice. It includes the customer, invoice date, billing location, and total invoice amount.

In [0]:
%sql
SELECT *
FROM read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/Invoice.csv',
    format => 'csv',
    header => true,
    inferSchema => true
)
LIMIT 10;

## Invoice Line Data

The InvoiceLine file contains the individual tracks purchased in each invoice. This is the most detailed sales source and is the likely basis of the sales fact table.

In [0]:
%sql
SELECT *
FROM read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/InvoiceLine.csv',
    format => 'csv',
    header => true,
    inferSchema => true
)
LIMIT 10;

## Source Row Counts

Row counts provide a baseline that can be compared with the Raw, Clean, and Mart layers later.

In [0]:
%sql
SELECT 'Customer' AS source_file, COUNT(*) AS row_count
FROM read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/Customer.csv',
    format => 'csv',
    header => true,
    inferSchema => true
)

UNION ALL

SELECT 'Employee', COUNT(*)
FROM read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/Employee.csv',
    format => 'csv',
    header => true,
    inferSchema => true
)

UNION ALL

SELECT 'Invoice', COUNT(*)
FROM read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/Invoice.csv',
    format => 'csv',
    header => true,
    inferSchema => true
)

UNION ALL

SELECT 'InvoiceLine', COUNT(*)
FROM read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/InvoiceLine.csv',
    format => 'csv',
    header => true,
    inferSchema => true
)

ORDER BY source_file;

## Music Table Relationships

The Chinook music data is divided into several related tables:

- A track belongs to one album.
- An album belongs to one artist.
- A track belongs to one genre.
- A track uses one media type.

These tables are connected using AlbumId, ArtistId, GenreId, and MediaTypeId.

In [0]:
SELECT
    t.TrackId,
    t.Name AS track_name,
    al.Title AS album_title,
    ar.Name AS artist_name,
    g.Name AS genre_name,
    mt.Name AS media_type
FROM read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/Track.csv',
    format => 'csv',
    header => true,
    inferSchema => true
) t
LEFT JOIN read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/Album.csv',
    format => 'csv',
    header => true,
    inferSchema => true
) al
    ON t.AlbumId = al.AlbumId
LEFT JOIN read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/Artist.csv',
    format => 'csv',
    header => true,
    inferSchema => true
) ar
    ON al.ArtistId = ar.ArtistId
LEFT JOIN read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/Genre.csv',
    format => 'csv',
    header => true,
    inferSchema => true
) g
    ON t.GenreId = g.GenreId
LEFT JOIN read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/MediaType.csv',
    format => 'csv',
    header => true,
    inferSchema => true
) mt
    ON t.MediaTypeId = mt.MediaTypeId
LIMIT 10;

## Proposed Sales Fact Table Grain

The proposed grain of `fact_sales` is:

> One row represents one track purchased as part of an invoice.

InvoiceLine is the best starting point because it contains the most detailed sales records.

The future fact table will connect to:

- Customer dimension through CustomerId
- Date dimension through InvoiceDate
- Employee dimension through the customer's SupportRepId
- Track dimension through TrackId

Possible measures include:

- Quantity
- Unit price
- Sales amount calculated as Quantity × UnitPrice

In [0]:
SELECT
    InvoiceLineId,
    InvoiceId,
    TrackId,
    UnitPrice,
    Quantity,
    UnitPrice * Quantity AS calculated_sales
FROM read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/InvoiceLine.csv',
    format => 'csv',
    header => true,
    inferSchema => true
)
LIMIT 10;

## Initial Data-Quality Checks

These checks identify possible issues before loading the source files into the Raw layer.

The checks focus on:

- Duplicate primary IDs
- Missing important relationship IDs
- Consistency between invoice totals and invoice-line sales

In [0]:
SELECT
    'Customer' AS source_table,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CustomerId) AS unique_ids,
    COUNT(*) - COUNT(DISTINCT CustomerId) AS duplicate_ids
FROM read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/Customer.csv',
    format => 'csv',
    header => true,
    inferSchema => true
)

UNION ALL

SELECT
    'Employee',
    COUNT(*),
    COUNT(DISTINCT EmployeeId),
    COUNT(*) - COUNT(DISTINCT EmployeeId)
FROM read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/Employee.csv',
    format => 'csv',
    header => true,
    inferSchema => true
)

UNION ALL

SELECT
    'Invoice',
    COUNT(*),
    COUNT(DISTINCT InvoiceId),
    COUNT(*) - COUNT(DISTINCT InvoiceId)
FROM read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/Invoice.csv',
    format => 'csv',
    header => true,
    inferSchema => true
)

UNION ALL

SELECT
    'InvoiceLine',
    COUNT(*),
    COUNT(DISTINCT InvoiceLineId),
    COUNT(*) - COUNT(DISTINCT InvoiceLineId)
FROM read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/InvoiceLine.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

In [0]:
SELECT
    'Invoice missing CustomerId' AS quality_check,
    COUNT(*) AS issue_count
FROM read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/Invoice.csv',
    format => 'csv',
    header => true,
    inferSchema => true
)
WHERE CustomerId IS NULL

UNION ALL

SELECT
    'InvoiceLine missing InvoiceId',
    COUNT(*)
FROM read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/InvoiceLine.csv',
    format => 'csv',
    header => true,
    inferSchema => true
)
WHERE InvoiceId IS NULL

UNION ALL

SELECT
    'InvoiceLine missing TrackId',
    COUNT(*)
FROM read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/InvoiceLine.csv',
    format => 'csv',
    header => true,
    inferSchema => true
)
WHERE TrackId IS NULL

UNION ALL

SELECT
    'Track missing AlbumId',
    COUNT(*)
FROM read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/Track.csv',
    format => 'csv',
    header => true,
    inferSchema => true
)
WHERE AlbumId IS NULL;

In [0]:
SELECT
    i.InvoiceId,
    i.Total AS invoice_total,
    SUM(il.UnitPrice * il.Quantity) AS calculated_total,
    ROUND(i.Total - SUM(il.UnitPrice * il.Quantity), 2) AS difference
FROM read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/Invoice.csv',
    format => 'csv',
    header => true,
    inferSchema => true
) i
JOIN read_files(
    '/Volumes/workspace/bronze/1st_volume/shared/week05/chinook_csv/InvoiceLine.csv',
    format => 'csv',
    header => true,
    inferSchema => true
) il
    ON i.InvoiceId = il.InvoiceId
GROUP BY
    i.InvoiceId,
    i.Total
HAVING ROUND(i.Total - SUM(il.UnitPrice * il.Quantity), 2) <> 0;

## Source Exploration Findings

The initial source inspection was completed successfully.

### Findings

- All expected Chinook CSV files are available.
- Customer, Employee, Invoice, and InvoiceLine primary IDs are unique.
- The important relationship IDs checked are not missing.
- Invoice totals match the sum of their invoice-line sales.
- The normalized music tables can be connected through their existing IDs.
- InvoiceLine provides the correct detail level for the sales fact table.

### Proposed Data Flow

`Chinook CSV files → Raw tables → Clean tables → Dimensions and Fact table → Analytics → Dashboard`

### Proposed Fact Grain

One row in `fact_sales` will represent one track purchased as part of an invoice.

### Next Step

Read the original Chinook CSV files directly from the shared Databricks Volume and create Raw Delta tables in workspace.d3_raw. The source files will not be moved or modified.